# BRCA Spatial Transcriptomics — Notebook 3
## Niche analysis, morphology bridge, and final figures

Notebooks 1–2 gave every spot a **domain** and a **cell-type composition**. This notebook
turns the qualitative observations into **quantitative spatial statistics**, and connects
molecular state back to tissue *appearance* (the H&E image):

1. **Domain neighborhood enrichment** — which domains physically border each other?
2. **Cell-type co-localization** — do tumor and immune cells sit together or avoid each
   other? (quantifies the T-cell exclusion seen in Notebook 2)
3. **Co-occurrence vs distance** — how does immune presence change with distance from tumor?
4. **Morphology bridge** — extract features from the H&E image and relate tissue *texture*
   to molecular domains.
5. Final figures.

> Run on Colab. No GPU needed here (statistics + image features, not deep learning).

## 0 · Setup & load Notebook-2 output

In [ ]:
!pip install -q scanpy squidpy

import warnings; warnings.filterwarnings("ignore")
import os, numpy as np, pandas as pd
import scanpy as sc, squidpy as sq
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive")
WORKDIR = "/content/drive/MyDrive/brca_spatial"     # same as NB1/NB2
DECONV_H5AD = os.path.join(WORKDIR, "brca_visium_deconvolved.h5ad")
print("deconvolved file present?", os.path.exists(DECONV_H5AD))

In [ ]:
adata = sc.read_h5ad(DECONV_H5AD)
# proportion columns added in NB2 (one obs column per cell type)
celltypes = list(adata.obsm["proportions"].columns) if "proportions" in adata.obsm \
            else [c for c in adata.obs.columns if c in
                  ["B-cells","CAFs","Cancer Epithelial","Endothelial","Myeloid",
                   "Normal Epithelial","PVL","Plasmablasts","T-cells"]]
print(adata)
print("\ncell types:", celltypes)
print("domains:", sorted(adata.obs['domain'].unique(), key=int))

## 1 · Spatial graph
Rebuild the hexagonal neighbor graph on this object (same as Notebook 1: 6 neighbors).
All neighborhood statistics below run on this graph.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="grid", n_neighs=6)
print("spatial graph built:", adata.obsp["spatial_connectivities"].shape)

## 2 · Domain neighborhood enrichment — which domains border each other?

`nhood_enrichment` asks, for every pair of domains, whether spots of one sit next to spots
of the other **more (or less) often than random**. It permutes the labels many times to
build a null distribution and reports a **z-score**:

- large **positive** z → the two domains border each other more than chance (co-localized)
- large **negative** z → they *avoid* each other (segregated)

The diagonal (a domain with itself) is always strongly positive — domains are internally
contiguous. The off-diagonal is the interesting part.

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="domain", seed=0)
sq.pl.nhood_enrichment(adata, cluster_key="domain", figsize=(7, 6),
                       title="Domain neighborhood enrichment (z-score)")

## 3 · Cell-type co-localization — tumor vs immune vs stroma

Neighborhood enrichment needs *categorical* labels, but cell types are *continuous*
proportions. So we assign each spot its **dominant cell type** (the argmax of its
proportions) and run the same enrichment on that. This quantifies the visual story from
Notebook 2: do Cancer Epithelial and T-cells border each other, or avoid each other?

In [ ]:
# dominant cell type per spot
prop = adata.obsm["proportions"]
adata.obs["dominant_celltype"] = pd.Categorical(prop.columns[np.asarray(prop).argmax(1)])
print(adata.obs["dominant_celltype"].value_counts())

In [ ]:
# spatial map of the dominant cell type
sq.pl.spatial_scatter(adata, color="dominant_celltype", size=1.4, figsize=(7, 7),
                      title="Dominant cell type per spot")

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="dominant_celltype", seed=0)
sq.pl.nhood_enrichment(adata, cluster_key="dominant_celltype", figsize=(7, 6),
                       title="Cell-type neighborhood enrichment (z-score)")

**How to read it for the exclusion question:** find the row for *Cancer Epithelial*
and the column for *T-cells*. A strongly **negative** z-score is quantitative evidence that
T-cells are spatially **excluded** from the tumor — the "cold tumor" pattern we saw
qualitatively. A positive Plasmablasts–Plasmablasts diagonal with little else confirms the
isolated plasma-cell aggregate.

## 4 · Co-occurrence vs distance — how far does the exclusion reach?

`co_occurrence` measures, starting from spots of a reference type, how the probability of
each other type changes with **distance**. Anchoring on *Cancer Epithelial*, an immune type
whose curve *drops* with proximity (low near tumor, higher far away) is the signature of
exclusion; one that *rises* near the tumor is infiltration.

In [ ]:
sq.gr.co_occurrence(adata, cluster_key="dominant_celltype")
anchor = "Cancer Epithelial" if "Cancer Epithelial" in \
         adata.obs["dominant_celltype"].cat.categories else \
         adata.obs["dominant_celltype"].cat.categories[0]
sq.pl.co_occurrence(adata, cluster_key="dominant_celltype", clusters=anchor, figsize=(7, 4))
plt.suptitle(f"Co-occurrence vs distance from: {anchor}"); plt.show()

## 5 · Morphology bridge (H&E) — tissue *appearance* vs molecular state

> **Experimental section.** This extracts features from the histology image. The squidpy
> image API is sensitive to versions and to how the image/scale factors are stored, so if a
> cell here errors, tell me the message — the niche analysis above is the validated core and
> stands on its own.

The link to the thesis: we ask whether *how the tissue looks* (H&E texture/intensity) tracks
*what it is molecularly* (the domains). We extract simple per-spot image features and test
whether they differ across domains.

In [ ]:
# build an ImageContainer from the H&E stored in the AnnData
library_id = list(adata.uns["spatial"].keys())[0]
sf = adata.uns["spatial"][library_id]["scalefactors"]
img = sq.im.ImageContainer(
    adata.uns["spatial"][library_id]["images"]["hires"],
    scale=sf["tissue_hires_scalef"],
)
print("image container:", img.shape)

In [ ]:
# per-spot summary + texture features from the H&E
sq.im.calculate_image_features(
    adata, img, features=["summary", "texture"],
    key_added="morphology", show_progress_bar=False,
)
morph = adata.obsm["morphology"]
print("extracted", morph.shape[1], "image features per spot")
morph.head()

In [ ]:
# do image features differ across molecular domains?
# mean of each feature per domain, z-scored across domains for readability
feat = adata.obsm["morphology"].copy()
feat["domain"] = adata.obs["domain"].values
by_dom = feat.groupby("domain", observed=True).mean()
by_dom_z = (by_dom - by_dom.mean()) / (by_dom.std() + 1e-9)

plt.figure(figsize=(10, 4))
plt.imshow(by_dom_z.values, aspect="auto", cmap="coolwarm", vmin=-2, vmax=2)
plt.yticks(range(by_dom_z.shape[0]), [f"dom {d}" for d in by_dom_z.index])
plt.xticks(range(by_dom_z.shape[1]), by_dom_z.columns, rotation=90, fontsize=7)
plt.colorbar(label="z-scored feature"); plt.title("H&E image features per domain")
plt.tight_layout(); plt.show()

If rows (domains) show distinct feature patterns, then tissue *morphology* carries a
signature of molecular identity — i.e. the way a region looks under H&E reflects its
cell-type composition. That is the molecular echo of the thesis idea that structure and
state are coupled.

## 6 · Save final object

In [ ]:
out = os.path.join(WORKDIR, "brca_visium_final.h5ad")
adata.write(out)
print("saved ->", out)

---
### Project complete (analysis side)
You now have: GNN spatial domains (NB1), per-spot cell-type composition (NB2), and
quantitative niche structure + a morphology link (NB3). Next steps for the repo: export the
key figures to `figures/`, finish the README results, and (optional) the robustness checks
listed under *Limitations* — domain stability across seeds, and correlating domains against
technical covariates.